<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260320_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JWT(JSON Web Token)

## 1. JWT의 정의와 등장 배경

### 개념

JWT는 정보를 안전하게 전송하기 위해 서명된 토큰입니다. 서명되어 있으므로 전송되는 정보의 신뢰성을 검증할 수 있습니다. **JWT(JSON Web Token)**는 당사자 간에 정보를 JSON 개체로 안전하게 전송하기 위한 개방형 표준(RFC 7519)입니다. 인증 및 정보 교류를 위해 웹 개발에서 필수적으로 사용되는 개념입니다.

### 등장 배경 (Session vs Token)

- **세션 기반 인증**: 서버 메모리나 DB에 세션 상태를 저장해야 하므로 서버 확장이 어렵고 메모리 부담이 큽니다.
- **토큰 기반 인증 (JWT)**: 토큰 자체에 정보를 담고 있어 서버가 상태를 유지할 필요가 없는 **Stateless** 방식입니다.

## 2. JWT의 구조

JWT는 점(.)으로 구분된 세 부분으로 구성됩니다. 실제 JWT 토큰은 아래와 같이 **점(.)으로 연결된 세 덩어리의 문자열** 형태입니다.

실제 JWT 토큰 예시

> **eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9**.**eyJ1c2VySWQiOiJhcHBsZSIsInVzZXJOYW1lIjoia2ltIiwiaWF0IjoxNTE2MjM5MDIyfQ**.**SflKxwRJSMeKKF2QT4fwpMeJf36POk6yJV_adQssw5c**
>

위의 긴 문자열을 점(`.`) 기준으로 나누면 다음과 같은 의미를 담고 있습니다.

1. **Header** : `eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9`
    - 해독하면: `{"alg": "HS256", "typ": "JWT"}`
    - **의미**: "이건 HMAC SHA256 알고리즘을 사용한 JWT 토큰이야."
2. **Payload** : `eyJ1c2VySWQiOiJhcHBsZSIsInVzZXJOYW1lIjoia2ltIiwiaWF0IjoxNTE2MjM5MDIyfQ`
    - 해독하면: `{"userId": "apple", "userName": "kim", "iat": 1516239022}`
    - **의미**: "로그인한 사람은 apple이고 이름은 kim이야. (iat는 생성 시간)"
3. **Signature** : `SflKxwRJSMeKKF2QT4fwpMeJf36POk6yJV_adQssw5c`
    - **의미**: "서버만 아는 비밀키로 위 두 부분을 섞어서 만든 '디지털 도장'이야. 누군가 내용을 1글자라도 바꾸면 이 도장이랑 안 맞게 돼!"

### 1) Header (헤더)

토큰의 유형(typ)과 사용 중인 서명 알고리즘(alg)을 정의합니다.

- 예: `{"alg": "HS256", "typ": "JWT"}`

### 2) Payload (페이로드)

토큰에 담긴 정보인 Claim(클레임)을 포함합니다. 클레임은 세 종류로 나뉩니다.

- **Registered claims**: 서비스에서 필요한 정보를 저장하기 위해 이미 정의된 속성 (iss: 발행자, exp: 만료시간, sub: 제목 등)
- **Public claims**: 충돌 방지를 위해 URI 형식으로 정의된 속성
- **Private claims**: 양 당사자 간의 협의 하에 사용되는 사용자 정의 속성

### 3) Signature (서명)

헤더의 인코딩 값과 페이로드의 인코딩 값을 합친 후, 서버가 가진 비밀키(Secret Key)로 해싱하여 생성합니다. 이를 통해 메시지가 변조되지 않았음을 확인합니다.

## 4. JWT의 장점과 단점

### 장점

- **무상태성(Stateless)**: 서버 확장이 용이하며, 별도의 세션 저장소가 필요 없습니다.
- **보안성**: 디지털 서명이 포함되어 데이터 변조를 방지할 수 있습니다.
- **범용성**: HTTP 헤더를 통해 전달되므로 어떤 플랫폼(Web, Mobile)에서도 동작합니다.

### 단점 및 주의사항

- **데이터 크기**: 페이로드에 많은 정보를 담을수록 토큰이 길어져 네트워크 부하가 생길 수 있습니다.
- **수정 불가**: 한 번 발행된 토큰은 만료 전까지 정보를 수정하거나 강제로 무효화하기 어렵습니다.
- **보안 노출**: 페이로드는 단순 Base64 인코딩이므로 누구나 내용을 볼 수 있습니다. **중요한 민감 정보(비밀번호 등)는 절대 포함하면 안 됩니다.**

## 5. 보안 강화 전략

### Access Token & Refresh Token

JWT의 탈취 위험을 줄이기 위해 두 종류의 토큰을 함께 사용합니다.

- **Access Token**: 수명이 짧으며(예: 30분), 실제 인증에 사용됩니다.
- **Refresh Token**: 수명이 길며(예: 2주), Access Token이 만료되었을 때 새로운 토큰을 발급받는 용도로 사용됩니다.



---



In [38]:
%%bash
apt install -y mariadb-server mariadb-client -q
service mariadb start
mysql -u root -e "
CREATE DATABASE IF NOT EXISTS testdb;
CREATE USER IF NOT EXISTS 'testuser'@'localhost' IDENTIFIED BY '1234';
GRANT ALL PRIVILEGES ON testdb.* TO 'testuser'@'localhost';
FLUSH PRIVILEGES;
SELECT 1;
"

Reading package lists...
Building dependency tree...
Reading state information...
mariadb-client is already the newest version (1:10.6.23-0ubuntu0.22.04.1).
mariadb-server is already the newest version (1:10.6.23-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.
 * Starting MariaDB database server mariadbd
   ...done.
1
1




---



In [39]:
%%bash
mkdir -p /content/chungjung/static /content/chungjung/templates



---



In [40]:
%%writefile /content/chungjung/package.json

{
  "name": "chungjung-ilbo",
  "main": "server.js",
  "dependencies": {
    "express": "^4.18.2",
    "mysql2": "^3.6.0"
  }
}

Overwriting /content/chungjung/package.json


In [41]:
%%writefile /content/chungjung/static/style.css

* { box-sizing: border-box; margin: 0; padding: 0; }

body {
    background: #ffffff;
    color: #1a1a1a;
    font-family: 'Apple SD Gothic Neo', 'Noto Sans KR', 'Malgun Gothic', Arial, sans-serif;
    line-height: 1.6;
    font-size: 14px;
}

a { color: inherit; text-decoration: none; }

.newspaper-header {
    background: #fff;
    border-bottom: 3px solid #005c2e;
}

.header-inner {
    max-width: 1280px;
    margin: 0 auto;
    padding: 0 24px;
}

.header-logo {
    display: flex;
    justify-content: center;
    padding: 18px 0 14px;
    border-bottom: 1px solid #eee;
}

.header-top {
    display: flex;
    justify-content: flex-end;
    padding: 6px 0;
    border-bottom: 1px solid #eee;
}

.header-meta-right {
    font-size: 12px;
    color: #999;
    text-align: right;
    line-height: 1.8;
}

.header-nav {
    display: flex;
    padding: 0;
}

.header-nav a {
    padding: 10px 18px;
    font-size: 14px;
    font-weight: 600;
    color: #333;
    border-right: 1px solid #eee;
    transition: color 0.15s;
}

.header-nav a:first-child { border-left: 1px solid #eee; }
.header-nav a:hover, .header-nav a.active { color: #005c2e; }

.page {
    max-width: 1280px;
    margin: 0 auto;
    padding: 24px 24px 60px;
}

.section-header {
    display: flex;
    align-items: center;
    gap: 8px;
    margin-bottom: 16px;
    padding-bottom: 10px;
    border-bottom: 2px solid #1a1a1a;
}

.section-label { font-size: 16px; font-weight: 800; color: #1a1a1a; }
.section-date { font-size: 12px; color: #999; margin-left: auto; }

.news-grid {
    display: grid;
    grid-template-columns: repeat(12, 1fr);
    border: 1px solid #e8e8e8;
}

.card-headline {
    grid-column: span 8;
    border-right: 1px solid #e8e8e8;
    padding: 24px 28px;
    background: #fff;
    cursor: pointer;
    transition: background 0.1s;
}
.card-headline:hover { background: #fafafa; }

.news-sidebar {
    grid-column: span 4;
    display: flex;
    flex-direction: column;
}

.card-side {
    flex: 1;
    padding: 16px 20px;
    background: #fff;
    border-bottom: 1px solid #e8e8e8;
    cursor: pointer;
    transition: background 0.1s;
}
.card-side:last-child { border-bottom: none; }
.card-side:hover { background: #fafafa; }

.news-grid-bottom {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    border: 1px solid #e8e8e8;
    border-top: none;
}

.card-normal {
    padding: 18px 22px;
    background: #fff;
    border-right: 1px solid #e8e8e8;
    cursor: pointer;
    transition: background 0.1s;
}
.card-normal:last-child { border-right: none; }
.card-normal:hover { background: #fafafa; }

.card-category {
    display: inline-block;
    font-size: 11px;
    font-weight: 700;
    color: #005c2e;
    margin-bottom: 8px;
    letter-spacing: 0.5px;
}

.card-title {
    font-size: 20px;
    font-weight: 700;
    line-height: 1.4;
    margin-bottom: 10px;
    color: #1a1a1a;
    letter-spacing: -0.3px;
}

.card-headline .card-title { font-size: 26px; letter-spacing: -0.5px; }
.card-side .card-title { font-size: 15px; }
.card-normal .card-title { font-size: 16px; }

.card-content {
    font-size: 13px;
    color: #666;
    line-height: 1.75;
    display: -webkit-box;
    -webkit-line-clamp: 3;
    -webkit-box-orient: vertical;
    overflow: hidden;
}

.card-headline .card-content { font-size: 14px; color: #444; -webkit-line-clamp: 5; }
.card-side .card-content { font-size: 12px; -webkit-line-clamp: 2; }

.card-meta {
    font-size: 11px;
    color: #aaa;
    margin-top: 12px;
    display: flex;
    justify-content: space-between;
    padding-top: 10px;
    border-top: 1px solid #f0f0f0;
}

.card-divider {
    border: none;
    border-top: 2px solid #005c2e;
    margin: 10px 0 14px;
    width: 36px;
}

.empty-state {
    grid-column: 1 / -1;
    text-align: center;
    padding: 80px 20px;
    color: #bbb;
    font-size: 14px;
    background: #fafafa;
}

.form-page {
    max-width: 720px;
    margin: 0 auto;
    padding: 32px 24px 60px;
}

.form-header {
    margin-bottom: 24px;
    padding-bottom: 14px;
    border-bottom: 2px solid #1a1a1a;
}

.form-header h1 {
    font-size: 24px;
    font-weight: 800;
    letter-spacing: -0.5px;
}

.form-header p { font-size: 13px; color: #999; margin-top: 6px; }

.form-panel {
    background: #fff;
    border: 1px solid #e8e8e8;
    padding: 28px;
}

.form-group { margin-bottom: 18px; }

label {
    display: block;
    font-size: 12px;
    font-weight: 700;
    color: #555;
    margin-bottom: 7px;
}

input[type="text"], select, textarea {
    width: 100%;
    border: 1px solid #ddd;
    border-radius: 3px;
    padding: 10px 12px;
    font-size: 14px;
    font-family: 'Apple SD Gothic Neo', 'Noto Sans KR', sans-serif;
    color: #1a1a1a;
    background: #fff;
    outline: none;
    transition: border-color 0.15s;
}

input[type="text"]:focus, select:focus, textarea:focus { border-color: #005c2e; }

textarea { min-height: 200px; resize: vertical; line-height: 1.8; }

.button-row { display: flex; gap: 8px; margin-top: 20px; }

.btn {
    flex: 1;
    padding: 11px;
    border: none;
    cursor: pointer;
    font-size: 14px;
    font-weight: 700;
    font-family: inherit;
    border-radius: 3px;
    transition: opacity 0.15s;
}
.btn:hover { opacity: 0.85; }
.btn-primary { background: #005c2e; color: #fff; }
.btn-secondary { background: #f5f5f5; color: #555; border: 1px solid #ddd; }
.btn-danger { background: #333; color: #fff; }

.modal-overlay {
    display: none;
    position: fixed;
    inset: 0;
    background: rgba(0,0,0,0.5);
    z-index: 1000;
    justify-content: center;
    align-items: flex-start;
    padding: 40px 20px;
    overflow-y: auto;
}
.modal-overlay.show { display: flex; }

.modal {
    background: #fff;
    max-width: 700px;
    width: 100%;
    padding: 36px;
    position: relative;
    border-top: 3px solid #005c2e;
    box-shadow: 0 8px 32px rgba(0,0,0,0.12);
}

.modal-close {
    position: absolute;
    top: 14px;
    right: 18px;
    font-size: 20px;
    cursor: pointer;
    color: #bbb;
    border: none;
    background: none;
    line-height: 1;
}
.modal-close:hover { color: #1a1a1a; }

.modal-category {
    font-size: 11px;
    font-weight: 700;
    color: #005c2e;
    letter-spacing: 0.5px;
    margin-bottom: 10px;
}

.modal-title {
    font-size: 23px;
    font-weight: 800;
    line-height: 1.35;
    margin-bottom: 12px;
    letter-spacing: -0.5px;
    padding-bottom: 14px;
    border-bottom: 1px solid #eee;
}

.modal-meta {
    font-size: 12px;
    color: #aaa;
    margin-bottom: 20px;
    display: flex;
    gap: 14px;
}

.modal-content {
    font-size: 15px;
    line-height: 1.9;
    color: #333;
    white-space: pre-wrap;
    word-break: break-word;
}

.modal-actions {
    display: flex;
    gap: 8px;
    margin-top: 24px;
    padding-top: 14px;
    border-top: 1px solid #eee;
}
.modal-actions .btn { flex: none; padding: 9px 18px; }

.toast {
    position: fixed;
    bottom: 24px;
    right: 24px;
    min-width: 220px;
    padding: 11px 16px;
    background: #1a1a1a;
    color: #fff;
    font-size: 13px;
    font-weight: 600;
    border-left: 3px solid #005c2e;
    opacity: 0;
    transform: translateY(8px);
    pointer-events: none;
    transition: opacity 0.2s, transform 0.2s;
    z-index: 9999;
    border-radius: 2px;
}
.toast.show { opacity: 1; transform: translateY(0); }
.toast.error { border-left-color: #ff4444; }

.loading {
    grid-column: 1 / -1;
    text-align: center;
    padding: 60px;
    color: #ccc;
    font-size: 13px;
}

Overwriting /content/chungjung/static/style.css


In [42]:
%%writefile /content/chungjung/server.js

// ── 필요한 라이브러리 불러오기 ──────────────────────────────
const express = require('express');        // 웹 서버 프레임워크
const path = require('path');              // 파일 경로 처리
const mysql = require('mysql2/promise');   // MariaDB 연결 (async/await 지원)
const jwt = require('jsonwebtoken');       // JWT 토큰 생성·검증
const bcrypt = require('bcrypt');          // 비밀번호 암호화

const app = express();
const PORT = 3000;
const JWT_SECRET = 'chungjung_secret_key_2026'; // 토큰 서명에 쓰는 비밀키 (절대 외부 노출 금지)

// ── 미들웨어 설정 ──────────────────────────────
app.use(express.json());                          // req.body에서 JSON 데이터 파싱
app.use(express.urlencoded({ extended: true }));  // HTML form 데이터 파싱
app.use('/static', express.static(path.join(__dirname, 'static'))); // /static 경로로 CSS·이미지 제공

// ── DB 연결 풀 생성 ──────────────────────────────
// 매 요청마다 연결을 새로 만들면 느리므로, 연결을 미리 여러 개 만들어두고 재사용
const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb',
    waitForConnections: true, // 연결이 모두 사용 중이면 대기
    connectionLimit: 10,      // 최대 동시 연결 수
    queueLimit: 0             // 대기열 제한 없음
});

// ── 서버 시작 시 DB 연결 확인 ──────────────────────────────
async function checkDbConnection() {
    try {
        const conn = await pool.getConnection(); // 풀에서 연결 하나 꺼냄
        console.log('DB 연결 성공');
        conn.release(); // 사용 후 풀에 반납
    } catch (err) {
        console.error('DB 연결 실패:', err);
    }
}
checkDbConnection();

// ── JWT 검증 미들웨어 ──────────────────────────────
// 인증이 필요한 라우트 앞에 붙이면, 토큰이 없거나 가짜면 여기서 바로 막힘
function verifyToken(req, res, next) {
    // Authorization 헤더에서 토큰 추출 (형식: "Bearer 토큰값")
    const token = req.headers['authorization']?.split(' ')[1];

    // 토큰이 아예 없으면 401 응답
    if (!token) return res.status(401).json({ success: false, message: '로그인이 필요합니다.' });

    // 토큰이 있으면 비밀키로 검증
    jwt.verify(token, JWT_SECRET, (err, decoded) => {
        if (err) return res.status(401).json({ success: false, message: '토큰이 유효하지 않습니다.' });

        req.user = decoded; // 검증 성공 → 해독된 유저 정보를 req에 담아 다음으로 전달
        next();             // 다음 미들웨어(라우트 핸들러)로 이동
    });
}

// ── 페이지 라우트 ──────────────────────────────
// 각 URL 접속 시 해당 HTML 파일 전송
app.get('/', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'index.html')));
app.get('/login', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'login.html')));
app.get('/signup', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'signup.html')));
app.get('/write', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'write.html')));
app.get('/edit/:id', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'edit.html')));

// ── 인증 API ──────────────────────────────

// [회원가입] POST /auth/signup
app.post('/auth/signup', async (req, res) => {
    try {
        const { userId, password, userName } = req.body; // 클라이언트가 보낸 데이터 꺼냄
        if (!userId || !password || !userName)
            return res.status(400).json({ success: false, message: '모든 항목을 입력해주세요.' });

        const hashed = await bcrypt.hash(password, 10); // 비밀번호를 해시로 암호화 (10=보안강도)
        await pool.query(
            'INSERT INTO users (user_id, password, user_name) VALUES (?, ?, ?)',
            [userId, hashed, userName] // ? 자리에 순서대로 대입 (SQL 인젝션 방어)
        );
        res.json({ success: true });
    } catch (err) {
        // UNIQUE 제약으로 중복 아이디면 INSERT 실패 → 여기서 잡힘
        res.status(500).json({ success: false, message: '이미 사용 중인 아이디입니다.' });
    }
});

// [로그인] POST /auth/login
app.post('/auth/login', async (req, res) => {
    try {
        const { userId, password } = req.body;

        // DB에서 해당 아이디 유저 조회
        const [rows] = await pool.query('SELECT * FROM users WHERE user_id = ?', [userId]);
        const user = rows[0]; // 결과 첫 번째 행

        // 유저가 존재하고, 입력 비밀번호가 암호화된 비밀번호와 일치하면
        if (user && await bcrypt.compare(password, user.password)) {
            // JWT 생성: payload(유저 정보) + 비밀키 + 만료시간 1시간
            const token = jwt.sign(
                { userId: user.user_id, userName: user.user_name },
                JWT_SECRET,
                { expiresIn: '1h' }
            );
            res.json({ success: true, accessToken: token }); // 토큰을 클라이언트에 전달
        } else {
            res.status(401).json({ success: false, message: '아이디 또는 비밀번호가 틀렸습니다.' });
        }
    } catch (err) {
        res.status(500).json({ success: false, message: '서버 오류가 발생했습니다.' });
    }
});

// [토큰 검증] GET /auth/verify
// 클라이언트가 "내 토큰 아직 유효해?" 확인할 때 사용
app.get('/auth/verify', (req, res) => {
    const token = req.headers['authorization']?.split(' ')[1];
    if (!token) return res.json({ success: false });

    jwt.verify(token, JWT_SECRET, (err, decoded) => {
        if (err) return res.json({ success: false }); // 만료됐거나 위조된 토큰
        res.json({ success: true, user: decoded });   // 유효하면 유저 정보 돌려줌
    });
});

// ── 뉴스 API ──────────────────────────────

// [기사 목록 조회] GET /api/news (인증 불필요 - 누구나 볼 수 있음)
app.get('/api/news', async (req, res) => {
    try {
        const [rows] = await pool.query(
            'SELECT id, title, category, author, created_at FROM news ORDER BY id DESC' // 최신순 정렬
        );
        res.json({ success: true, data: rows });
    } catch (err) {
        res.status(500).json({ success: false, message: '기사 목록을 불러오는 데 실패했습니다.' });
    }
});

// [기사 단건 조회] GET /api/news/:id (인증 불필요)
app.get('/api/news/:id', async (req, res) => {
    try {
        const [rows] = await pool.query('SELECT * FROM news WHERE id = ?', [req.params.id]);
        if (rows.length === 0)
            return res.status(404).json({ success: false, message: '기사를 찾을 수 없습니다.' });
        res.json({ success: true, data: rows[0] });
    } catch (err) {
        res.status(500).json({ success: false, message: '기사 조회 중 오류가 발생했습니다.' });
    }
});

// [기사 송고] POST /api/news (인증 필요 - verifyToken 미들웨어 통과해야 실행됨)
app.post('/api/news', verifyToken, async (req, res) => {
    try {
        const { title, category, content, author } = req.body;
        if (!title || !category || !content || !author)
            return res.status(400).json({ success: false, message: '모든 항목을 입력해주세요.' });

        await pool.query(
            'INSERT INTO news (title, category, content, author) VALUES (?, ?, ?, ?)',
            [title, category, content, author]
        );
        res.json({ success: true, message: '기사가 송고되었습니다.' });
    } catch (err) {
        res.status(500).json({ success: false, message: '기사 송고 중 오류가 발생했습니다.' });
    }
});

// [기사 수정] PUT /api/news/:id (인증 필요)
app.put('/api/news/:id', verifyToken, async (req, res) => {
    try {
        const { title, category, content, author } = req.body;
        const [result] = await pool.query(
            'UPDATE news SET title=?, category=?, content=?, author=? WHERE id=?',
            [title, category, content, author, req.params.id]
        );
        if (result.affectedRows === 0) // 해당 id가 없으면 수정된 행이 0
            return res.status(404).json({ success: false, message: '기사를 찾을 수 없습니다.' });
        res.json({ success: true, message: '기사가 수정되었습니다.' });
    } catch (err) {
        res.status(500).json({ success: false, message: '기사 수정 중 오류가 발생했습니다.' });
    }
});

// [기사 삭제] DELETE /api/news/:id (인증 필요)
app.delete('/api/news/:id', verifyToken, async (req, res) => {
    try {
        const [result] = await pool.query('DELETE FROM news WHERE id=?', [req.params.id]);
        if (result.affectedRows === 0)
            return res.status(404).json({ success: false, message: '기사를 찾을 수 없습니다.' });
        res.json({ success: true, message: '기사가 삭제되었습니다.' });
    } catch (err) {
        res.status(500).json({ success: false, message: '기사 삭제 중 오류가 발생했습니다.' });
    }
});

// ── 서버 시작 ──────────────────────────────
app.listen(PORT, () => {
    console.log('========================================');
    console.log(` 충정일보 서버 실행 중: http://localhost:${PORT}`);
    console.log('========================================');
});

Overwriting /content/chungjung/server.js


In [43]:
%%writefile /content/chungjung/templates/index.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>충정일보</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>

<header class="newspaper-header">
    <div class="header-inner">
        <div class="header-logo">
            <img src="https://raw.githubusercontent.com/flathfk/Bootcamp_Study/main/chungjung_news_logo.png" alt="충정일보" style="height:100px;">
        </div>
        <div class="header-top">
            <div class="header-meta-right">
                <div id="today-date"></div>
                <div>대한민국 대표 디지털 신문 · 제 1호</div>
            </div>
        </div>
        <nav class="header-nav">
            <a href="/" class="active">전체</a>
            <a href="/write" id="writeLink">기사 송고</a>
            <a href="/login" id="loginLink" style="display:none;">로그인</a>
            <a href="#" id="logoutLink" style="display:none;" onclick="logout()">로그아웃</a>
            <span id="userNameDisplay" style="padding:10px 18px; font-size:14px; font-weight:700; color:#005c2e; display:none;"></span>
        </nav>
    </div>
</header>

<div class="page">
    <div class="section-header">
        <span class="section-label">최신 기사</span>
        <span class="section-date" id="today-date-sub"></span>
    </div>

    <!-- 메인 그리드 (헤드라인 + 사이드) -->
    <div class="news-grid" id="mainGrid">
        <div class="loading">기사를 불러오는 중...</div>
    </div>

    <!-- 하단 일반 기사 그리드 -->
    <div class="news-grid-bottom" id="bottomGrid" style="display:none;"></div>
</div>

<!-- 기사 상세 모달 -->
<div class="modal-overlay" id="modalOverlay">
    <div class="modal">
        <button class="modal-close" onclick="closeModal()">✕</button>
        <div class="modal-category" id="modalCategory"></div>
        <h2 class="modal-title" id="modalTitle"></h2>
        <div class="modal-meta">
            <span id="modalAuthor"></span>
            <span id="modalDate"></span>
        </div>
        <div class="modal-content" id="modalContent"></div>
        <div class="modal-actions">
            <button class="btn btn-secondary" onclick="closeModal()">닫기</button>
            <button class="btn btn-primary" id="modalEditBtn">수정</button>
            <button class="btn btn-danger" id="modalDeleteBtn">삭제</button>
        </div>
    </div>
</div>

<div class="toast" id="toast"></div>

<script>
    // 오늘 날짜
    document.getElementById('today-date').textContent =
        new Date().toLocaleDateString('ko-KR', { year:'numeric', month:'long', day:'numeric', weekday:'long' });

    // 로그인 상태 확인 및 헤더 업데이트
    async function checkAuth() {
        const token = localStorage.getItem('accessToken');
        const loginLink = document.getElementById('loginLink');
        const logoutLink = document.getElementById('logoutLink');
        const userNameDisplay = document.getElementById('userNameDisplay');

        if (!token) {
            loginLink.style.display = 'inline-block';
            return false;
        }

        const res = await fetch('/auth/verify', {
            headers: { 'Authorization': `Bearer ${token}` }
        });
        const data = await res.json();

        if (data.success) {
            userNameDisplay.textContent = data.user.userName + ' 기자';
            userNameDisplay.style.display = 'inline-block';
            logoutLink.style.display = 'inline-block';
            return true;
        } else {
            localStorage.removeItem('accessToken');
            loginLink.style.display = 'inline-block';
            return false;
        }
    }

    function logout() {
        localStorage.removeItem('accessToken');
        location.reload();
    }

    function getToken() {
        return localStorage.getItem('accessToken');
    }

    // 토스트
    function showToast(message, isError = false) {
        const toast = document.getElementById('toast');
        toast.textContent = message;
        toast.className = isError ? 'toast show error' : 'toast show';
        setTimeout(() => { toast.className = isError ? 'toast error' : 'toast'; }, 2500);
    }

    // 기사 목록 로드
    async function loadNews() {
        try {
            const res = await fetch('/api/news');
            const result = await res.json();

            if (!result.success) throw new Error(result.message);

            renderNews(result.data);
        } catch (err) {
            document.getElementById('mainGrid').innerHTML =
                '<div class="empty-state">기사를 불러오지 못했습니다.</div>';
        }
    }

    function escapeHtml(str) {
        if (!str) return '';
        return String(str)
            .replace(/&/g, '&amp;')
            .replace(/</g, '&lt;')
            .replace(/>/g, '&gt;')
            .replace(/"/g, '&quot;');
    }

    function formatDate(str) {
        if (!str) return '';
        return new Date(str).toLocaleDateString('ko-KR', { month:'long', day:'numeric', hour:'2-digit', minute:'2-digit' });
    }

    function renderNews(articles) {
        const mainGrid = document.getElementById('mainGrid');
        const bottomGrid = document.getElementById('bottomGrid');

        if (articles.length === 0) {
            mainGrid.innerHTML = '<div class="empty-state">아직 등록된 기사가 없습니다.<br>첫 번째 기사를 송고해보세요.</div>';
            bottomGrid.style.display = 'none';
            return;
        }

        // 헤드라인: 첫 번째 기사
        const headline = articles[0];
        let mainHtml = `
            <div class="card-headline" onclick="openModal(${headline.id})">
                <div class="card-category">${escapeHtml(headline.category)}</div>
                <hr class="card-divider">
                <div class="card-title">${escapeHtml(headline.title)}</div>
                <div class="card-content">${escapeHtml(headline.content)}</div>
                <div class="card-meta">
                    <span>기자 ${escapeHtml(headline.author)}</span>
                    <span>${formatDate(headline.created_at)}</span>
                </div>
            </div>
        `;

        // 사이드바: 2~4번째 기사 (최대 3개)
        const sideArticles = articles.slice(1, 4);
        if (sideArticles.length > 0) {
            let sideHtml = '<div class="news-sidebar">';
            sideArticles.forEach(a => {
                sideHtml += `
                    <div class="card-side" onclick="openModal(${a.id})">
                        <div class="card-category">${escapeHtml(a.category)}</div>
                        <div class="card-title">${escapeHtml(a.title)}</div>
                        <div class="card-content">${escapeHtml(a.content)}</div>
                        <div class="card-meta">
                            <span>${escapeHtml(a.author)}</span>
                            <span>${formatDate(a.created_at)}</span>
                        </div>
                    </div>
                `;
            });
            sideHtml += '</div>';
            mainHtml += sideHtml;
        }

        mainGrid.innerHTML = mainHtml;

        // 하단 그리드: 5번째 이후 기사
        const bottomArticles = articles.slice(4);
        if (bottomArticles.length > 0) {
            bottomGrid.style.display = 'grid';
            bottomGrid.innerHTML = bottomArticles.map(a => `
                <div class="card-normal" onclick="openModal(${a.id})">
                    <div class="card-category">${escapeHtml(a.category)}</div>
                    <div class="card-title" style="font-size:17px;">${escapeHtml(a.title)}</div>
                    <div class="card-content">${escapeHtml(a.content)}</div>
                    <div class="card-meta">
                        <span>${escapeHtml(a.author)}</span>
                        <span>${formatDate(a.created_at)}</span>
                    </div>
                </div>
            `).join('');
        } else {
            bottomGrid.style.display = 'none';
        }
    }

    // 모달 열기
    let currentArticleId = null;

    async function openModal(id) {
        try {
            const res = await fetch(`/api/news/${id}`);
            const result = await res.json();
            if (!result.success) throw new Error(result.message);

            const a = result.data;
            currentArticleId = id;

            document.getElementById('modalCategory').textContent = a.category;
            document.getElementById('modalTitle').textContent = a.title;
            document.getElementById('modalAuthor').textContent = `기자 ${a.author}`;
            document.getElementById('modalDate').textContent = formatDate(a.created_at);
            document.getElementById('modalContent').textContent = a.content;
            document.getElementById('modalEditBtn').onclick = () => { window.location.href = `/edit/${id}`; };
            document.getElementById('modalDeleteBtn').onclick = () => deleteArticle(id);

            document.getElementById('modalOverlay').classList.add('show');
        } catch (err) {
            showToast('기사를 불러오지 못했습니다.', true);
        }
    }

    function closeModal() {
        document.getElementById('modalOverlay').classList.remove('show');
        currentArticleId = null;
    }

    // 모달 외부 클릭 시 닫기
    document.getElementById('modalOverlay').addEventListener('click', function(e) {
        if (e.target === this) closeModal();
    });

    // 삭제
    async function deleteArticle(id) {
        if (!confirm('이 기사를 삭제하시겠습니까?')) return;

        const token = getToken();
        if (!token) { showToast('로그인이 필요합니다.', true); return; }

        try {
            const res = await fetch(`/api/news/${id}`, {
                method: 'DELETE',
                headers: { 'Authorization': `Bearer ${token}` }
            });
            const result = await res.json();

            if (!result.success) throw new Error(result.message);

            closeModal();
            showToast(result.message);
            await loadNews();
        } catch (err) {
            showToast(err.message, true);
        }
    }

    // 초기 로드
    checkAuth();
    loadNews();
</script>
</body>
</html>

Overwriting /content/chungjung/templates/index.html


In [44]:
%%writefile /content/chungjung/templates/write.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>기사 송고 - 충정일보</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>

<header class="newspaper-header">
    <div class="header-inner">
        <div class="header-logo">
            <img src="https://raw.githubusercontent.com/flathfk/Bootcamp_Study/main/chungjung_news_logo.png" alt="충정일보" style="height:100px;">
        </div>
        <div class="header-top">
            <div class="header-meta-right">
                <div>대한민국 대표 디지털 신문 · 제 1호</div>
            </div>
        </div>
        <nav class="header-nav">
            <a href="/">전체</a>
            <a href="/write" class="active">기사 송고</a>
        </nav>
    </div>
</header>

<div class="form-page">
    <div class="form-header">
        <h1>기사 송고</h1>
        <p>작성한 기사를 충정일보 데스크로 송고합니다.</p>
    </div>

    <div class="form-panel">
        <form id="writeForm">
            <div class="form-group">
                <label>제목</label>
                <input type="text" name="title" placeholder="기사 제목을 입력하세요" required>
            </div>

            <div class="form-group">
                <label>카테고리</label>
                <select name="category" required>
                    <option value="">카테고리 선택</option>
                    <option value="정치">정치</option>
                    <option value="경제">경제</option>
                    <option value="사회">사회</option>
                    <option value="국제">국제</option>
                    <option value="연예">연예</option>
                    <option value="스포츠">스포츠</option>
                    <option value="IT">IT</option>
                    <option value="문화">문화</option>
                </select>
            </div>

            <div class="form-group">
                <label>기자명</label>
                <!-- readonly: 직접 수정 불가, 로그인한 기자 이름이 JS로 자동 입력됨 -->
                <input type="text" id="author" name="author" readonly style="background:#f5f5f5;">
            </div>

            <div class="form-group">
                <label>본문</label>
                <textarea name="content" placeholder="기사 본문을 입력하세요..." required></textarea>
            </div>

            <div class="button-row">
                <button class="btn btn-primary" type="submit">송고하기</button>
                <a class="btn btn-secondary" href="/" style="text-align:center; display:flex; align-items:center; justify-content:center;">취소</a>
            </div>
        </form>
    </div>
</div>

<div class="toast" id="toast"></div>

<script>
    // ── 로그인 상태 확인 ──────────────────────────────
    async function checkLogin() {
        const token = localStorage.getItem('accessToken'); // 브라우저 저장소에서 토큰 꺼냄

        // 토큰 없으면 로그인 페이지로 강제 이동
        if (!token) { location.href = '/login'; return; }

        // 서버에 토큰이 아직 유효한지 확인 요청
        // Authorization 헤더에 "Bearer 토큰값" 형식으로 전송
        const res = await fetch('/auth/verify', { headers: { 'Authorization': `Bearer ${token}` } });
        const data = await res.json();

        // 토큰 만료됐거나 위조된 경우 → 저장소에서 지우고 로그인 페이지로
        if (!data.success) { localStorage.removeItem('accessToken'); location.href = '/login'; }

        // 검증 성공 → 토큰 안에 있던 유저 이름을 기자명 칸에 자동 입력
        document.getElementById('author').value = data.user.userName;
    }
    checkLogin(); // 페이지 로드 즉시 실행

    // ── 토스트 메시지 (송고 성공/실패 알림) ──────────────────────────────
    function showToast(message, isError = false) {
        const toast = document.getElementById('toast');
        toast.textContent = message;
        toast.className = isError ? 'toast show error' : 'toast show';
        setTimeout(() => { toast.className = isError ? 'toast error' : 'toast'; }, 2500);
    }

    // ── 기사 송고 폼 제출 ──────────────────────────────
    document.getElementById('writeForm').addEventListener('submit', async function(e) {
        e.preventDefault(); // 기본 form 제출(페이지 이동) 막기

        const token = localStorage.getItem('accessToken');
        if (!token) { location.href = '/login'; return; } // 혹시 토큰 없으면 차단

        // form 입력값을 객체로 변환
        // FormData → Object.fromEntries → { title: '...', category: '...', ... }
        const formData = new FormData(this);
        const payload = Object.fromEntries(formData.entries());

        try {
            const res = await fetch('/api/news', {
                method: 'POST',
                headers: {
                    'Content-Type': 'application/json',   // 내가 보내는 데이터가 JSON임을 서버에 알림
                    'Authorization': `Bearer ${token}`    // 서버의 verifyToken 미들웨어가 이걸 확인함
                },
                body: JSON.stringify(payload) // 객체를 JSON 문자열로 변환해서 전송
            });

            const result = await res.json();

            if (!result.success) throw new Error(result.message); // 실패 시 catch로 이동

            showToast(result.message);
            setTimeout(() => { window.location.href = '/'; }, 800); // 0.8초 후 메인으로 이동
        } catch (err) {
            showToast(err.message, true); // 에러 토스트 출력
        }
    });
</script>
</body>
</html>

Overwriting /content/chungjung/templates/write.html


In [55]:
%%writefile /content/chungjung/templates/edit.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>기사 수정 - 충정일보</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>

<header class="newspaper-header">
    <div class="header-inner">
        <div class="header-logo">
            <img src="https://raw.githubusercontent.com/flathfk/Bootcamp_Study/main/chungjung_news_logo.png" alt="충정일보" style="height:100px;">
        </div>
        <div class="header-top">
            <div class="header-meta-right">
                <div>대한민국 대표 디지털 신문 · 제 1호</div>
            </div>
        </div>
        <nav class="header-nav">
            <a href="/">전체</a>
            <a href="/write">기사 송고</a>
        </nav>
    </div>
</header>

<div class="form-page">
    <div class="form-header">
        <h1>기사 수정</h1>
        <p>기사 내용을 수정합니다.</p>
    </div>

    <div class="form-panel">
        <form id="editForm">
            <div class="form-group">
                <label>제목</label>
                <input type="text" id="title" name="title" required>
            </div>

            <div class="form-group">
                <label>카테고리</label>
                <select id="category" name="category" required>
                    <option value="정치">정치</option>
                    <option value="경제">경제</option>
                    <option value="사회">사회</option>
                    <option value="국제">국제</option>
                    <option value="연예">연예</option>
                    <option value="스포츠">스포츠</option>
                    <option value="IT">IT</option>
                    <option value="문화">문화</option>
                </select>
            </div>

            <div class="form-group">
                <label>기자명</label>
                <input type="text" id="author" name="author" required>
            </div>

            <div class="form-group">
                <label>본문</label>
                <textarea id="content" name="content" required></textarea>
            </div>

            <div class="button-row">
                <button class="btn btn-primary" type="submit">수정 완료</button>
                <a class="btn btn-secondary" href="/" style="text-align:center; display:flex; align-items:center; justify-content:center;">취소</a>
            </div>
        </form>
    </div>
</div>

<div class="toast" id="toast"></div>

<script>
    // URL에서 기사 id 추출 (/edit/3 → "3")
    const articleId = window.location.pathname.split('/').pop();

    // ── 로그인 상태 확인 ──────────────────────────────
    async function checkLogin() {
        const token = localStorage.getItem('accessToken');

        // 토큰 없으면 로그인 페이지로 강제 이동
        if (!token) { location.href = '/login'; return; }

        // 서버에 토큰 유효성 확인
        const res = await fetch('/auth/verify', { headers: { 'Authorization': `Bearer ${token}` } });
        const data = await res.json();

        // 토큰 만료 또는 위조 → 저장소에서 지우고 로그인 페이지로
        if (!data.success) { localStorage.removeItem('accessToken'); location.href = '/login'; }
    }
    checkLogin(); // 페이지 로드 즉시 실행

    // ── 토스트 메시지 ──────────────────────────────
    function showToast(message, isError = false) {
        const toast = document.getElementById('toast');
        toast.textContent = message;
        toast.className = isError ? 'toast show error' : 'toast show';
        setTimeout(() => { toast.className = isError ? 'toast error' : 'toast'; }, 2500);
    }

    // ── 기존 기사 데이터 불러오기 ──────────────────────────────
    // 수정 페이지 열릴 때 현재 저장된 내용을 폼에 미리 채워줌
    async function loadArticle() {
        try {
            const res = await fetch(`/api/news/${articleId}`); // 단건 조회 API 호출
            const result = await res.json();

            if (!result.success) throw new Error(result.message);

            const a = result.data;
            document.getElementById('title').value = a.title;     // 제목 채우기
            document.getElementById('author').value = a.author;   // 기자명 채우기
            document.getElementById('content').value = a.content; // 본문 채우기

            // 카테고리 select에서 현재 저장된 값과 일치하는 옵션 선택 상태로 만들기
            const categorySelect = document.getElementById('category');
            for (const opt of categorySelect.options) {
                if (opt.value === a.category) { opt.selected = true; break; }
            }
        } catch (err) {
            showToast('기사 정보를 불러오지 못했습니다.', true);
        }
    }

    // ── 수정 폼 제출 ──────────────────────────────
    document.getElementById('editForm').addEventListener('submit', async function(e) {
        e.preventDefault(); // 기본 form 제출(페이지 이동) 막기

        const token = localStorage.getItem('accessToken');
        if (!token) { location.href = '/login'; return; }

        // form 입력값을 객체로 변환
        const formData = new FormData(this);
        const payload = Object.fromEntries(formData.entries());

        try {
            // PUT 요청: 기존 기사를 수정 (id 포함)
            const res = await fetch(`/api/news/${articleId}`, {
                method: 'PUT',
                headers: {
                    'Content-Type': 'application/json',
                    'Authorization': `Bearer ${token}` // 서버 verifyToken 미들웨어가 확인
                },
                body: JSON.stringify(payload)
            });

            const result = await res.json();
            if (!result.success) throw new Error(result.message);

            showToast(result.message);
            setTimeout(() => { window.location.href = '/'; }, 800); // 0.8초 후 메인으로 이동
        } catch (err) {
            showToast(err.message, true);
        }
    });

    loadArticle(); // 페이지 로드 시 기존 데이터 채우기
</script>
</body>
</html>

Overwriting /content/chungjung/templates/edit.html


In [46]:
%%writefile /content/chungjung/templates/login.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>충정일보 - 로그인</title>
    <link rel="stylesheet" href="/static/style.css">
    <style>
        body { background: #f5f5f5; display: flex; flex-direction: column; min-height: 100vh; }
        .login-wrap { flex: 1; display: flex; justify-content: center; align-items: center; padding: 40px 20px; }
        .login-box { background: #fff; border: 1px solid #e8e8e8; border-top: 3px solid #005c2e; padding: 40px; width: 100%; max-width: 400px; }
        .login-title { font-size: 18px; font-weight: 800; text-align: center; margin-bottom: 24px; color: #1a1a1a; }
        .form-group { margin-bottom: 14px; }
        .form-group label { display: block; font-size: 12px; font-weight: 700; color: #555; margin-bottom: 6px; }
        .form-group input { width: 100%; border: 1px solid #ddd; border-radius: 3px; padding: 11px 12px; font-size: 14px; outline: none; box-sizing: border-box; }
        .form-group input:focus { border-color: #005c2e; }
        .btn-login { width: 100%; padding: 12px; background: #005c2e; color: #fff; border: none; border-radius: 3px; font-size: 14px; font-weight: 700; cursor: pointer; margin-top: 6px; }
        .btn-login:hover { opacity: 0.88; }
        .signup-link { text-align: center; margin-top: 16px; font-size: 13px; color: #999; }
        .signup-link a { color: #005c2e; font-weight: 700; }
        .error-msg { font-size: 13px; color: #c0392b; margin-top: 8px; text-align: center; display: none; }
    </style>
</head>
<body>

<header class="newspaper-header">
    <div class="header-inner">
        <div class="header-logo">
            <img src="https://raw.githubusercontent.com/flathfk/Bootcamp_Study/main/chungjung_news_logo.png" alt="충정일보" style="height:70px;">
        </div>
    </div>
</header>

<div class="login-wrap">
    <div class="login-box">
        <div class="login-title">기자 로그인</div>

        <div class="form-group">
            <label>아이디</label>
            <input type="text" id="userId" placeholder="아이디 입력">
        </div>
        <div class="form-group">
            <label>비밀번호</label>
            <!-- onkeydown: 엔터키 눌렀을 때 login() 실행 (버튼 안 눌러도 됨) -->
            <input type="password" id="password" placeholder="비밀번호 입력" onkeydown="if(event.key==='Enter') login()">
        </div>

        <div class="error-msg" id="errorMsg">아이디 또는 비밀번호가 틀렸습니다.</div>

        <button class="btn-login" onclick="login()">로그인</button>

        <div class="signup-link">
            계정이 없으신가요? <a href="/signup">회원가입</a>
        </div>
    </div>
</div>

<script>
    // ── 이미 로그인 상태면 메인으로 자동 이동 ──────────────────────────────
    // 로그인 페이지를 굳이 볼 필요 없으므로 토큰이 있으면 바로 내보냄
    const token = localStorage.getItem('accessToken');
    if (token) {
        fetch('/auth/verify', { headers: { 'Authorization': `Bearer ${token}` } })
            .then(r => r.json())
            .then(d => { if (d.success) location.href = '/'; }); // 유효한 토큰이면 메인으로
    }

    // ── 로그인 함수 ──────────────────────────────
    async function login() {
        const userId = document.getElementById('userId').value.trim();
        const password = document.getElementById('password').value;
        const errorMsg = document.getElementById('errorMsg');

        // 입력값 검증: 빈 칸이면 에러 메시지 보여주고 중단
        if (!userId || !password) {
            errorMsg.textContent = '아이디와 비밀번호를 입력해주세요.';
            errorMsg.style.display = 'block';
            return;
        }

        try {
            // 서버 로그인 API에 아이디/비밀번호 전송
            const res = await fetch('/auth/login', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ userId, password })
            });
            const data = await res.json();

            if (data.success) {
                // 로그인 성공 → 서버가 준 JWT를 브라우저 저장소에 보관
                localStorage.setItem('accessToken', data.accessToken);
                location.href = '/'; // 메인 페이지로 이동
            } else {
                // 로그인 실패 → 에러 메시지 표시
                errorMsg.textContent = data.message || '로그인에 실패했습니다.';
                errorMsg.style.display = 'block';
            }
        } catch (err) {
            // 네트워크 오류 등 예상치 못한 에러
            errorMsg.textContent = '서버 오류가 발생했습니다.';
            errorMsg.style.display = 'block';
        }
    }
</script>
</body>
</html>

Overwriting /content/chungjung/templates/login.html


In [47]:
%%writefile /content/chungjung/templates/signup.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>충정일보 - 회원가입</title>
    <link rel="stylesheet" href="/static/style.css">
    <style>
        body { background: #f5f5f5; display: flex; flex-direction: column; min-height: 100vh; }
        .login-wrap { flex: 1; display: flex; justify-content: center; align-items: center; padding: 40px 20px; }
        .login-box { background: #fff; border: 1px solid #e8e8e8; border-top: 3px solid #005c2e; padding: 40px; width: 100%; max-width: 400px; }
        .login-title { font-size: 18px; font-weight: 800; text-align: center; margin-bottom: 24px; color: #1a1a1a; }
        .form-group { margin-bottom: 14px; }
        .form-group label { display: block; font-size: 12px; font-weight: 700; color: #555; margin-bottom: 6px; }
        .form-group input { width: 100%; border: 1px solid #ddd; border-radius: 3px; padding: 11px 12px; font-size: 14px; outline: none; box-sizing: border-box; }
        .form-group input:focus { border-color: #005c2e; }
        .btn-login { width: 100%; padding: 12px; background: #005c2e; color: #fff; border: none; border-radius: 3px; font-size: 14px; font-weight: 700; cursor: pointer; margin-top: 6px; }
        .btn-login:hover { opacity: 0.88; }
        .signup-link { text-align: center; margin-top: 16px; font-size: 13px; color: #999; }
        .signup-link a { color: #005c2e; font-weight: 700; }
        .error-msg { font-size: 13px; color: #c0392b; margin-top: 8px; text-align: center; display: none; }
    </style>
</head>
<body>

<header class="newspaper-header">
    <div class="header-inner">
        <div class="header-logo">
            <img src="https://raw.githubusercontent.com/flathfk/Bootcamp_Study/main/chungjung_news_logo.png" alt="충정일보" style="height:70px;">
        </div>
    </div>
</header>

<div class="login-wrap">
    <div class="login-box">
        <div class="login-title">기자 회원가입</div>

        <div class="form-group">
            <label>아이디</label>
            <input type="text" id="userId" placeholder="아이디 입력">
        </div>
        <div class="form-group">
            <label>비밀번호</label>
            <input type="password" id="password" placeholder="비밀번호 입력">
        </div>
        <div class="form-group">
            <label>이름</label>
            <input type="text" id="userName" placeholder="기자 이름 입력">
        </div>

        <!-- 에러 메시지 영역: 기본은 숨김, JS에서 display:block으로 보여줌 -->
        <div class="error-msg" id="errorMsg"></div>

        <button class="btn-login" onclick="signup()">가입하기</button>

        <div class="signup-link">
            이미 계정이 있으신가요? <a href="/login">로그인</a>
        </div>
    </div>
</div>

<script>
    // ── 회원가입 함수 ──────────────────────────────
    async function signup() {
        // 입력값 가져오기
        // trim(): 앞뒤 공백 제거 (실수로 스페이스만 입력하는 경우 방지)
        const userId = document.getElementById('userId').value.trim();
        const password = document.getElementById('password').value;
        const userName = document.getElementById('userName').value.trim();
        const errorMsg = document.getElementById('errorMsg');

        // 입력값 검증: 하나라도 비어있으면 에러 메시지 표시 후 중단
        if (!userId || !password || !userName) {
            errorMsg.textContent = '모든 항목을 입력해주세요.';
            errorMsg.style.display = 'block';
            return;
        }

        try {
            // 서버 회원가입 API에 데이터 전송
            const res = await fetch('/auth/signup', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ userId, password, userName })
            });
            const data = await res.json();

            if (data.success) {
                // 가입 성공 → 로그인 페이지로 이동
                alert('가입이 완료되었습니다. 로그인해주세요.');
                location.href = '/login';
            } else {
                // 가입 실패 (주로 중복 아이디) → 에러 메시지 표시
                errorMsg.textContent = data.message || '회원가입에 실패했습니다.';
                errorMsg.style.display = 'block';
            }
        } catch (err) {
            // 네트워크 오류 등 예상치 못한 에러
            errorMsg.textContent = '서버 오류가 발생했습니다.';
            errorMsg.style.display = 'block';
        }
    }
</script>
</body>
</html>

Overwriting /content/chungjung/templates/signup.html




---



In [48]:
%%bash
cd /content/chungjung && npm install


removed 17 packages, and audited 81 packages in 1s

18 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities




---



In [49]:
%%bash
mysql -u root -e "
CREATE USER IF NOT EXISTS 'testuser'@'localhost' IDENTIFIED BY '1234';
GRANT ALL PRIVILEGES ON testdb.* TO 'testuser'@'localhost';
FLUSH PRIVILEGES;
"

mysql -u testuser -p1234 testdb -e "
CREATE TABLE IF NOT EXISTS news (
    id INT AUTO_INCREMENT PRIMARY KEY,
    title VARCHAR(300) NOT NULL,
    category VARCHAR(50) NOT NULL,
    content TEXT NOT NULL,
    author VARCHAR(100) NOT NULL,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS users (
    id INT AUTO_INCREMENT PRIMARY KEY,
    user_id VARCHAR(50) NOT NULL UNIQUE,
    password VARCHAR(255) NOT NULL,
    user_name VARCHAR(50) NOT NULL
);
"

In [50]:
%%bash
cd /content/chungjung
npm install express mysql2 bcrypt jsonwebtoken
npm install -g cloudflared


added 17 packages, and audited 98 packages in 2s

18 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities

changed 1 package in 2s


In [51]:
%%bash
cd /content/chungjung
nohup node server.js > /content/server.log 2>&1 &
sleep 2
cat /content/server.log

node:events:502
      throw er; // Unhandled 'error' event
      ^

Error: listen EADDRINUSE: address already in use :::3000
    at Server.setupListenHandle [as _listen2] (node:net:1908:16)
    at listenInCluster (node:net:1965:12)
    at Server.listen (node:net:2067:7)
    at Function.listen (/content/chungjung/node_modules/express/lib/application.js:635:24)
    at Object.<anonymous> (/content/chungjung/server.js:180:5)
    at Module._compile (node:internal/modules/cjs/loader:1529:14)
    at Module._extensions..js (node:internal/modules/cjs/loader:1613:10)
    at Module.load (node:internal/modules/cjs/loader:1275:32)
    at Module._load (node:internal/modules/cjs/loader:1096:12)
    at Function.executeUserEntryPoint [as runMain] (node:internal/modules/run_main:164:12)
Emitted 'error' event on Server instance at:
    at emitErrorNT (node:net:1944:8)
    at process.processTicksAndRejections (node:internal/process/task_queues:82:21) {
  code: 'EADDRINUSE',
  errno: -98,
  syscall: 'liste

In [52]:
%%bash
nohup cloudflared tunnel --url http://localhost:3000 > /content/tunnel.log 2>&1 &
sleep 5
cat /content/tunnel.log | grep trycloudflare.com

2026-03-20T07:22:16Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-20T07:22:19Z INF |  https://designation-evaluated-catalogs-sale.trycloudflare.com                             |




---



In [54]:
%%bash
cat /content/tunnel.log | grep trycloudflare.com

2026-03-20T07:22:16Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-20T07:22:19Z INF |  https://designation-evaluated-catalogs-sale.trycloudflare.com                             |
